In [1]:
import math
import random
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [44]:
class Value:
    def __init__(self, data, _children=(),_op='',label=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"Value(data={self.data})"
    

    def __add__(self, other):
        other = other if isinstance(other,Value) else Value(other)
        out = Value(self.data + other.data,(self , other), "+")

        def backward():
            self.grad += 1.0*out.grad
            other.grad += 1.0*out.grad
        out._backward = backward

        return out
    
    def __mul__(self,other):
        other = other if isinstance(other,Value) else Value(other)
        out = Value(self.data * other.data,(self,other), "*")

        def backward():
            self.grad += other.data*out.grad
            other.grad += self.data*out.grad
        out._backward = backward
        return out

    def __rmul__(self, other):
        return self * other

    def __radd__(self, other):
        return self + other
    
    def __rsub__(self, other):
        return other + (-self)
    
    def __rtruediv__(self, other):
        return other * self **-1
        
    def __pow__(self, other):
        assert isinstance(other, (int, float))
        out = Value(self.data**other,(self,), f'**({other})')
        def backward():
            self.grad += other*self.data**(other-1)*out.grad
        out._backward = backward
        return out

    def __truediv__(self,other):
        return self * other ** -1 

    def __neg__(self):
        return -1 * self

    def __sub__(self,other):
        return self + (-other)

    def tanh(self):
        n = self.data
        t = (math.exp(2*n) - 1) / (math.exp(2*n) + 1)
        out = Value(t , (self,), "tanh")
        def backward():
            self.grad += (1 - t**2)*out.grad
        out._backward= backward
        return out

    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()

    def exp(self):
        out = Value(math.exp(self.data), (self,), 'exp')
        def backward():
            self.grad += out.data * out.grad
        out._backward = backward

        return out

    def tanh_exp(self):
        x = (2*self).exp()
        out = (x-1)/(x+1)
        return out
    

In [3]:
a = Value(3.0); b = Value(-2.0)
c = a * b + a**2
c.backward()
assert a.grad == 4.0 ,f"a.grad == {a.grad}"
assert b.grad == 3.0 , f"b.grad == {b.grad}"
print("test 1 passed: add and mul gradients")

test 1 passed: add and mul gradients


In [4]:
a = Value(3.0); b = Value(-2.0)
c = a * b + a
c.backward()
assert a.grad == -1.0 ,f"a.grad == {a.grad}"
assert b.grad == 3.0 , f"b.grad == {b.grad}"
print("test 2 passed")

test 2 passed


In [11]:
a = Value(3.0)
c = a + a
c.backward()
assert a.grad == 2.0; f"a.grad == {a.grad}"
print("test 3 passed")

test 3 passed


In [6]:
a = Value (2.0); b = Value(2.0)
c = a ** 2
c.backward()
assert a.grad == 4.0; f"a.grad == {a.grad}"
d = b * b
d.backward()
assert b.grad == a.grad; f"b.grad == {b.grad}"
print("test 4 passed")

test 4 passed


In [7]:
a = Value(2.0); b = Value(-3.0)
c = a * b + b
d = c.tanh()
d.backward()
print(f"a.grad = {a.grad}")


a.grad = -1.8275975133263955e-07


In [9]:
x1 = Value(2.0)
x2 = Value(0.0)
w1 = Value(-3.0)
w2 = Value(1.0)
b = Value(6.8813735870195432)
n = x1*w1 + x2 * w2 + b
o = n.tanh()
o.backward()
print(f"""x1.grad = {x1.grad}, x2.grad = {x2.grad}
            w1.grad = {w1.grad}, w2.grad = {w2.grad},
            b.grad = {b.grad}""")

x1.grad = -1.4999999999999996, x2.grad = 0.4999999999999999
            w1.grad = 0.9999999999999998, w2.grad = 0.0,
            b.grad = 0.4999999999999999


In [10]:
import torch

In [31]:
x1 = torch.tensor([2.0]).double()         ;   x1.requires_grad = True
w1 = torch.tensor([-3.0]).double()        ;   w1.requires_grad = True
x2 = torch.tensor([0.0]).double()         ;   x2.requires_grad = True
w2 = torch.tensor([1.0]).double()         ;   w2.requires_grad = True
b = torch.tensor([6.8813735870195432], dtype= torch.float64)       ;   b.requires_grad = True
n = x1*w1 + x2 * w2 + b
o = n.tanh()
o.backward()

print(f"""
    {x1.grad.item()}, {w1.grad.item()}, {x2.grad.item()}, {w2.grad.item()}, {b.grad.item()}
""")



    -1.4999999999999996, 0.9999999999999998, 0.4999999999999999, 0.0, 0.4999999999999999



In [46]:
x1 = Value(2.0)
x2 = Value(0.0)
w1 = Value(-3.0)
w2 = Value(1.0)
b = Value(6.8813735870195432)
n = x1*w1 + x2 * w2 + b
o = n.tanh_exp()
o.backward()
print(f"""x1.grad = {x1.grad}, x2.grad = {x2.grad}
            w1.grad = {w1.grad}, w2.grad = {w2.grad},
            b.grad = {b.grad}""")

x1.grad = -1.5, x2.grad = 0.5
            w1.grad = 1.0, w2.grad = 0.0,
            b.grad = 0.5


In [72]:
class Neuron:
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1,1))

    def __call__(self,x):
        act = sum((wi*xi for wi, xi in zip(self.w , x)), self.b)
        out = act.tanh()
        return out

    def parameters(self):
        return self.w + [self.b]

class Layer:
    def __init__(self,nin,nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs

    def parameters(self):
        return [p for neuron in self.neurons for p in neuron.parameters()]

class MLP:
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

In [73]:
x = [3.0 , 2.0, -1.0]
n = MLP(3,[4,1])
n(x)

Value(data=-0.633713197892254)

In [79]:

xs = [[2.0, 3.0, -1.0], [3.0, -1.0, 0.5], [0.5, 1.0, 1.0], [1.0, 1.0, -1.0]]
ys = [1.0, -1.0, -1.0, 1.0]
n = MLP(3,[4,5,1])
n(x)

Value(data=0.9909619820631919)

In [80]:
for i in range(50):
    ypred = [n(x) for x in xs]
    loss = sum((ygt - yout)**2 for ygt , yout in zip(ys, ypred))

    for p in n.parameters():
        p.grad = 0.0
    loss.backward()

    for p in n.parameters():
        p.data += -0.06*p.grad
    print(i, loss.data)

0 7.888182938560229
1 7.872149858466722
2 7.850894104463846
3 7.821448651442495
4 7.778169753657141
5 7.709030852013545
6 7.584064945179305
7 7.310382841661958
8 6.516278864772931
9 4.198492021341558
10 2.407675627996093
11 1.321895172954384
12 0.43870091840303854
13 0.26212305535434
14 0.1949476339033134
15 0.15617367453517583
16 0.12954644676656146
17 0.11013399136091356
18 0.09539627430823139
19 0.08385934872030898
20 0.07460590392531889
21 0.06703526788830229
22 0.06073812445659355
23 0.05542634874932822
24 0.05089154188693216
25 0.046979442167131874
26 0.04357355503282003
27 0.040584355186043544
28 0.037941973305632884
29 0.035591126869798026
30 0.03348753381989523
31 0.03159532856714289
32 0.029885169422344073
33 0.02833283174554417
34 0.02691814798170989
35 0.025624199173441126
36 0.02443669129818994
37 0.023343469161961828
38 0.02233413386273347
39 0.021399739073529697
40 0.020532547908162457
41 0.019725836784200285
42 0.018973736059469982
43 0.018271099674937633
44 0.017613397

In [81]:
ypred

[Value(data=0.9356327706503192),
 Value(data=-0.9344140919937142),
 Value(data=-0.9384405876475125),
 Value(data=0.9486263793550237)]